# 01 · Salud del proyecto — Economía de La Rioja

Chequeo integral end-to-end del pipeline. Ejecutar con **Kernel → Restart & Run All**.

**Qué verifica:**
1. Setup de rutas e imports
2. Presencia de archivos fuente en `data/raw/`
3. Presencia de archivos procesados en `data/processed/`
4. Empresas EC (AFIP-DGIP) — serie 2019-2025
5. Empleo EC — serie 2019-2025
6. Benchmark 9 provincias (intensidad empresarial)
7. Prima salarial EC — 3 definiciones
8. DGIP — recaudación y contribuyentes
9. Empleo comparado LR / NOA / Nación
10. Resumen final con semáforos

**Definición vigente:** AFIP-DGIP (8 CIIU) — ver `src/empresas_ec.py`

## 1 · Setup

In [1]:
import os
import sys
import importlib
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

print("CWD:", Path.cwd())
print("Python:", sys.version.split()[0])
print("src OK:", (Path.cwd() / "src").exists())

for mod in ["empresas_ec", "prima_salarial", "regional", "dgip", "empleo", "sectorial"]:
    try:
        importlib.import_module(f"src.{mod}")
    except Exception as e:
        print(f"  [!] src.{mod}: {e}")

CWD: c:\Users\HP\Documents\Economy\economia_c
Python: 3.14.5
src OK: True
  Prima salarial EC — La Rioja

[1/3] Parseando remuneraciones...
[2/3] Cargando salario provincial...
[3/3] Calculando primas...

=== Prima salarial EC (% sobre salario provincial) ===
 anio  prima_nucleo_software  prima_nucleo_amplio  prima_afip_dgip_completo
 2019                   4.26               -27.93                    -13.99
 2020                  61.31                61.31                     29.62
 2021                  18.84                -7.48                    -17.79
 2022                  53.67                17.65                     -2.41
 2023                  20.04                 4.13                     -6.59
 2024                  45.75                45.75                     21.38
 2025                  29.00                29.00                      8.36

  prima_nucleo_software    → 7220 + 7250 + 7290 (recomendado dashboard)
  prima_nucleo_amplio      → + 7300 (I+D) + 7430 (publicida

## 2 · Archivos fuente (data/raw)

In [2]:
RAW = Path("data/raw")

CLAVE_RAW = [
    "provinciales_serie_empresas3_2.xlsx",
    "provinciales_serie_empleo_trimestral_4dig_7.xlsx",
    "provinciales_serie_remuneraciones_mensual_4dig_7.xlsx",
    "SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx",
    "SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx",
    "empresas_ec_rioja.csv",
    "total_provincial_rioja.csv",
]

print(f"{'Archivo':<60} {'Estado':>10} {'Tamaño':>12}")
print("-" * 84)
faltan_raw = []
for nombre in CLAVE_RAW:
    p = RAW / nombre
    if p.exists():
        kb = p.stat().st_size / 1024
        print(f"{nombre:<60} {'OK':>10} {kb:>10,.0f} KB")
    else:
        print(f"{nombre:<60} {'FALTA':>10} {'-':>12}")
        faltan_raw.append(nombre)

print(f"\nTotal archivos en data/raw: {len(list(RAW.iterdir()))}")
if faltan_raw:
    print(f"Faltan: {faltan_raw}")

Archivo                                                          Estado       Tamaño
------------------------------------------------------------------------------------
provinciales_serie_empresas3_2.xlsx                                  OK      1,171 KB
provinciales_serie_empleo_trimestral_4dig_7.xlsx                     OK      4,281 KB
provinciales_serie_remuneraciones_mensual_4dig_7.xlsx                OK     24,470 KB
SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx                       OK        139 KB
SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx                      OK        102 KB
empresas_ec_rioja.csv                                                OK          2 KB
total_provincial_rioja.csv                                           OK          0 KB

Total archivos en data/raw: 37


## 3 · Archivos procesados (data/processed)

In [3]:
PROCESSED = Path("data/processed")

CLAVE_PROC = [
    "empleo_ec_rioja.xlsx",
    "empresas_ec_rioja.xlsx",
    "empresas_ec_rioja_2022_2025.xlsx",
    "intensidad_provincias.csv",
    "prima_salarial_ec.xlsx",
]

print(f"{'Archivo':<45} {'Estado':>10} {'Tamaño':>12}")
print("-" * 70)
faltan_proc = []
for nombre in CLAVE_PROC:
    p = PROCESSED / nombre
    if p.exists():
        kb = p.stat().st_size / 1024
        print(f"{nombre:<45} {'OK':>10} {kb:>10,.0f} KB")
    else:
        print(f"{nombre:<45} {'ausente':>10} {'-':>12}")
        faltan_proc.append(nombre)

if faltan_proc:
    print(f"\nAusentes (pueden regenerarse): {faltan_proc}")

Archivo                                           Estado       Tamaño
----------------------------------------------------------------------
empleo_ec_rioja.xlsx                                  OK         33 KB
empresas_ec_rioja.xlsx                                OK          7 KB
empresas_ec_rioja_2022_2025.xlsx                      OK          8 KB
intensidad_provincias.csv                             OK          0 KB
prima_salarial_ec.xlsx                                OK          5 KB


## 4 · Empresas EC (AFIP-DGIP)

In [4]:
from src.empresas_ec import (
    cargar_ec, serie_ec, por_bloque, participacion_total,
    benchmark_provincias, CIIU_EC,
)

df_ec = cargar_ec()
serie = serie_ec(df_ec)

print("Serie anual de empresas EC:")
print(serie.to_string(index=False))

v2024 = serie.loc[serie["anio"] == 2024, "total_ec"].iloc[0]
assert v2024 == 102, f"Esperado 102 en 2024, encontre {v2024}"
print(f"\nOK: 2024 = {v2024:.0f} empresas")

part = participacion_total(df_ec)
print("\nParticipacion sobre total provincial:")
print(part.to_string(index=False))

Serie anual de empresas EC:
 anio  total_ec
 2019      70.0
 2020      65.0
 2021      74.0
 2022      95.0
 2024     102.0
 2025      84.0

OK: 2024 = 102 empresas

Participacion sobre total provincial:
 anio  total_ec  total_provincial  participacion_pct
 2019      70.0              2539               2.76
 2020      65.0              2416               2.69
 2021      74.0              2456               3.01
 2022      95.0              2873               3.31
 2024     102.0              3255               3.13
 2025      84.0              2935               2.86


## 5 · Empleo EC

In [5]:
import pandas as pd

empleo_ec = pd.read_excel("data/processed/empleo_ec_rioja.xlsx",
                           sheet_name="Empleo_EC_anual")
print("Empleo EC promedio anual:")
print(empleo_ec.to_string(index=False))

anios = empleo_ec["anio"].tolist()
assert 2019 in anios and 2025 in anios, "Faltan anios extremos"
print(f"\nOK: Serie completa {min(anios)}-{max(anios)} ({len(anios)} anios)")

Empleo EC promedio anual:
 anio  empleo_ec_prom
 1996          148.50
 1997          209.75
 1998          168.00
 1999          194.25
 2000          194.50
 2001          193.75
 2002          103.25
 2003          130.00
 2004          173.25
 2005          205.00
 2006          185.25
 2007          223.50
 2008          336.75
 2009          339.00
 2010          353.25
 2011          279.50
 2012          268.50
 2013          248.75
 2014          237.00
 2015          242.50
 2016          332.00
 2017          342.50
 2018          417.75
 2019          355.75
 2020          297.50
 2021          336.50
 2022          445.00
 2023          512.75
 2024          505.75
 2025          521.25

OK: Serie completa 1996-2025 (30 anios)


## 6 · Benchmark 9 provincias — intensidad EC 2024

In [6]:
bench = benchmark_provincias("2024")
print(bench.to_string(index=False))

lr = bench[bench["provincia"] == "La Rioja"]["intensidad_pm"].iloc[0]
assert 31.0 < lr < 32.0, f"La Rioja esperada ~31.3, encontre {lr}"
print(f"\nOK: La Rioja = {lr:.2f} por mil (esperado ~31.34)")

          provincia   ec  total  intensidad_pm
               CABA 9262 136305          67.95
          Catamarca  162   3752          43.18
              Salta  406  10412          38.99
            Tucumán  432  12086          35.74
            Córdoba 2011  58212          34.55
              Jujuy  177   5345          33.12
           La Rioja  102   3255          31.34
           Santa Fé 1473  53973          27.29
Santiago del Estero  163   6360          25.63

OK: La Rioja = 31.34 por mil (esperado ~31.34)


## 7 · Prima salarial EC — 3 definiciones

In [7]:
from src.prima_salarial import (
    parsear_hoja_remuneraciones, salario_provincial_mensual,
    RUTA_REMUN, RUTA_SAL_PROV, HOJA_REMUN,
    CIIU_NUCLEO_SW, CIIU_NUCLEO_AMPLIO, CIIU_COMPLETO,
)

df_remun = parsear_hoja_remuneraciones(RUTA_REMUN, HOJA_REMUN)
ec_remun = df_remun[df_remun["rama"].isin(CIIU_EC.keys())].copy()
sal_prov = salario_provincial_mensual(RUTA_SAL_PROV, "La Rioja")

def prima_mes(ciiu_set, etiqueta):
    filas = []
    anios_ok = [2019,2020,2021,2022,2023,2024,2025]
    for (a, m), g in ec_remun[ec_remun["anio"].isin(anios_ok)].groupby(["anio","mes"]):
        sub = g[g["rama"].isin(ciiu_set)]
        if sub.empty: continue
        sp_row = sal_prov[(sal_prov["anio"]==a) & (sal_prov["mes"]==m)]
        if sp_row.empty: continue
        sp = sp_row["valor"].iloc[0]
        if pd.isna(sp) or sp == 0: continue
        filas.append({"anio": a, "prima": (sub["remuneracion"].mean()/sp - 1)*100})
    if not filas:
        return pd.Series(dtype=float, name=etiqueta)
    return pd.DataFrame(filas).groupby("anio")["prima"].mean().round(2).rename(etiqueta)

sw = prima_mes(CIIU_NUCLEO_SW, "nucleo_software")
amp = prima_mes(CIIU_NUCLEO_AMPLIO, "nucleo_amplio")
comp = prima_mes(CIIU_COMPLETO, "afip_dgip_completo")

print(pd.concat([sw, amp, comp], axis=1).to_string())

v2025 = sw.loc[2025]
assert v2025 > 0, f"Prima nucleo software 2025 esperada >0, encontre {v2025}"
print(f"\nOK: Prima nucleo software 2025 = +{v2025:.2f}%")

      nucleo_software  nucleo_amplio  afip_dgip_completo
anio                                                    
2019             4.26         -27.93              -13.99
2020            61.31          61.31               29.62
2021            18.84          -7.48              -17.79
2022            53.67          17.65               -2.41
2023            20.04           4.13               -6.59
2024            45.75          45.75               21.38
2025            29.00          29.00                8.36

OK: Prima nucleo software 2025 = +29.00%


## 8 · DGIP — recaudación y contribuyentes

In [8]:
try:
    import src.dgip as dg
    bib = Path("reports/bibliografia")
    pdfs = sorted(bib.glob("dgip_*.pdf"))
    if pdfs:
        df_dgip = dg.cargar_dgip_tecnologico_multi(pdfs)
        tot = dg.dgip_total_anual(df_dgip)
        print("DGIP total anual:")
        print(tot.to_string(index=False))
    else:
        print("Sin PDFs dgip_*.pdf en reports/bibliografia - se omite")
except Exception as e:
    print(f"DGIP: {e}")

DGIP total anual:
 anio  contrib_conv  contrib_local    base_conv   base_local  impuesto_total
 2022          1752           1138 8.383045e+08 8.553079e+08    5.898752e+07
 2023          1911           1060 4.358462e+09 2.008145e+09    2.430125e+08
 2024          1947            844 1.472175e+10 5.232998e+09    8.461294e+08
 2025          2075            840 3.290781e+10 8.017545e+09    1.940909e+09


## 9 · Empleo comparado LR / NOA / Nación

In [9]:
try:
    from src.regional import serie_provincia, serie_nacional, serie_noa

    emp_lr = serie_provincia("La Rioja", "empleo")
    emp_nac = serie_nacional("empleo")
    emp_noa = serie_noa("empleo")

    print("Empleo LR (ultimos 3):")
    print(emp_lr.tail(3).to_string())
    print("\nEmpleo Nacion (ultimos 3):")
    print(emp_nac.tail(3).to_string())
    print("\nEmpleo NOA (ultimos 3):")
    print(emp_noa.tail(3).to_string())
    print("\nOK: Series LR/NOA/Nacion")
except Exception as e:
    print(f"Regional: {e}")

Empleo LR (ultimos 3):
fecha
2026-04-01    29.300
2026-05-01    29.188
2026-06-01    28.890

Empleo Nacion (ultimos 3):
fecha
2026-04-01    6141.363
2026-05-01    6132.641
2026-06-01    6126.380

Empleo NOA (ultimos 3):
fecha
2026-04-01    449.328
2026-05-01    448.869
2026-06-01    448.052

OK: Series LR/NOA/Nacion


## 10 · Resumen — semáforos del proyecto

In [10]:
print("=" * 60)
print("  RESUMEN DEL PROYECTO")
print("=" * 60)
print()
print(f"  Empresas EC 2024:         {v2024:.0f}  (esperado 102)")
print(f"  Intensidad LR 2024:       {lr:.2f} por mil  (esperado ~31.34)")
print(f"  Prima nucleo software:    +{sw.loc[2025]:.2f}%  (2025)")
print(f"  Prima AFIP-DGIP completa: +{comp.loc[2025]:.2f}%  (2025)")
print()
print(f"  data/raw OK:              {len(CLAVE_RAW) - len(faltan_raw)}/{len(CLAVE_RAW)}")
print(f"  data/processed OK:        {len(CLAVE_PROC) - len(faltan_proc)}/{len(CLAVE_PROC)}")
print()
print("=" * 60)
print("  Chequeo completado")
print("=" * 60)

  RESUMEN DEL PROYECTO

  Empresas EC 2024:         102  (esperado 102)
  Intensidad LR 2024:       31.34 por mil  (esperado ~31.34)
  Prima nucleo software:    +29.00%  (2025)
  Prima AFIP-DGIP completa: +8.36%  (2025)

  data/raw OK:              7/7
  data/processed OK:        5/5

  Chequeo completado


In [11]:
from pathlib import Path
p = Path("data/raw/total_provincial_rioja.csv")
print("Bytes:", p.stat().st_size)
print("Contenido crudo:")
print(p.read_text())

Bytes: 342
Contenido crudo:
anio,total_provincial
1996,1634
1997,1729
1998,1799
1999,1870
2000,1848
2001,1860
2002,1700
2003,1780
2004,1946
2005,2100
2006,2190
2007,2366
2008,2431
2009,2482
2010,2594
2011,2633
2012,2619
2013,2662
2014,2649
2015,2681
2016,2729
2017,2716
2018,2677
2019,2539
2020,2416
2021,2456
2022,2873
2024,3255
2025,2935

